### **Autor:** David Roca Tauste

---
---
# **📌ACTIVIDAD 3: ÁRBOLES DE DECISIÓN PARA REGRESIÓN**
---
---

Crea el notebook saa_u03_p01_a3-<tus_iniciales>.ipynb donde entregar esta actividad. Los árboles de decisión son propensos a presentar overfitting. En scikit-learn podemos reducirlo ajustando ciertos hiperparámetros. En el caso de DecisionTreeRegressor:
- Antes del entrenamiento:
    - Limitar la profundidad del árbol (max_depth)
    - Restringir el número mínimo de muestras por nodo (min_samples_split, min_samples_leaf)

- Poda después del entrenamiento:
    - Reducir la complejidad con "ccp_alfa".
    
Utiliza los modelos DecisionTreeRegressor, GradientBoostingRegressor, XGBRegressor y RandomForestRegressor para entrenarlos con los datos del ejercicio anterior intentando mejorar los resultados en caso de haberlos usado ya, o bajar el overfitting en caso de no haberlos usado.

## ENTREGA 7: Muestra Código, gráficos y capturas de ejecución de:
### a) Carga de datos y preprocesamiento (si es necesario).

In [13]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [14]:
dataset = pd.read_csv('centro-comercial.csv')
semilla = 546

In [15]:
dataset['fecha'] = pd.to_datetime(dataset['fecha'], format='%d-%m-%Y')

dataset['dia'] = dataset['fecha'].dt.day
dataset['mes'] = dataset['fecha'].dt.month
dataset['año'] = dataset['fecha'].dt.year

dataset = dataset.drop("fecha", axis=1)

In [16]:
variable_objetivo = "ventas_semanales"

target = dataset[variable_objetivo]
predictoras = dataset.drop(variable_objetivo, axis=1)

In [17]:
predictoras = predictoras.drop("IPC", axis=1)
dataset = dataset.drop("IPC", axis=1)

### b) Entrenamiento y configuración del DecisionTreeRegresor: desempeño inicial, cambio de hiperparámetros y desempeño final.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    predictoras, target, test_size=0.2, random_state=semilla
)

# Modelo Decision Tree con hiperparámetros
arbol = DecisionTreeRegressor(
    max_depth=5, 
    min_samples_split=10, 
    min_samples_leaf=5, 
    random_state=semilla
)
arbol.fit(X_train, y_train)

# Poda post-entrenamiento con coste-complejidad
camino = arbol.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = camino.ccp_alphas

# Elegir el mejor alpha basado en validación
arboles = [DecisionTreeRegressor(random_state=semilla, ccp_alpha=alpha) for alpha in ccp_alphas]
scores = [root_mean_squared_error(y_test, t.fit(X_train, y_train).predict(X_test)) for t in arboles]
mejor_alpha = ccp_alphas[np.argmin(scores)]

# Árbol con mejor "ccp_alfa"
arbol_tuneado = DecisionTreeRegressor(
    max_depth=5, 
    min_samples_split=10, 
    min_samples_leaf=5, 
    ccp_alpha=mejor_alpha, 
    random_state=semilla
)
arbol_tuneado.fit(X_train, y_train)

DecisionTreeRegressor(ccp_alpha=np.float64(16382979.039984822), max_depth=5,
                      min_samples_leaf=5, min_samples_split=10,
                      random_state=546)

### c) Igual para el GradientBoostingRegressor.

In [19]:
# Gradient Boosting Regressor
gbr = GradientBoostingRegressor(random_state=semilla)
gbr.fit(X_train, y_train)

GradientBoostingRegressor(random_state=546)

### d) Igual para XGBRegressor.

In [20]:
# XGBRegressor
xgb = XGBRegressor(random_state=semilla, verbosity=0)
xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=546, ...)

### e) Igual para RandomForestregressor.

In [21]:
# Random Forest Regressor
rf = RandomForestRegressor(random_state=semilla)
rf.fit(X_train, y_train)

RandomForestRegressor(random_state=546)

### f) Importancia o influencia de cada característica en alguno de los modelos.

In [22]:
# Comparación de errores
modelos = {
    "DecisionTree": arbol_tuneado,
    "Gradient Boosting": gbr,
    "XGBoost": xgb,
    "Random Forest": rf
}

In [24]:
print("RMSE\n----")

for nombre, modelo in modelos.items():
    pred = modelo.predict(X_test)
    error = root_mean_squared_error(y_test, pred)
    print(f"{nombre}: {error:.2f}")

RMSE
----
DecisionTree: 353306.18
Gradient Boosting: 190750.49
XGBoost: 78882.09
Random Forest: 121329.71
